In [9]:
%reload_ext autoreload
%autoreload 2

In [10]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import torch

from sckitflow import predict_adata
from sckitflow.core._types import PredictionData
from sckitflow.core.methods import ODEInference, ODEInferenceConfig
from sckitflow.core.nn import MLPVelocity
from sckitflow.core.surrogate import SurrogatePotential
from sckitflow.data import DataManagerConfig, FlowDataModule

In [ ]:
# ---------------------------------------------------------------------------
# Real data if we have it; a synthetic stand-in otherwise.
# ---------------------------------------------------------------------------
ADATA_PATH = Path("/Users/lorenzo.consoli/Work/data/collab-goettgens/loop0/data/adata_500k.h5ad")
N_OBS = 500
N_GENES = 19  # 19 + 7 scatter = 26, matching the state dim the notebook prints

MOLECULES = [
    "740-yp_[µm]",
    "mtg_[µm]",
    "rhflt3l_[ng_ml]",
    "gm-csf_[ng_ml]",
    "tpo_[ng_ml]",
    "sr1_[nm]",
    "um171_[nm]",
    "um729_[µm]",
    "scf_[ng_ml]",
    "butyzamide_[nm]",
    "retinoic_acid_[µm]",
    "ldl_[ng_ml]",
    "il3_[ng_ml]",
    "o2_[%]",
    "days_of_culture",
]
SCATTER_FEATURES = [
    "FSC-A :: FSC - Area",
    "FSC-H :: FSC - Height",
    "FSC-W :: FSC - Width",
    "SSC-A :: SSC - Area",
    "SSC-H :: SSC - Height",
    "SSC-W :: SSC - Width",
    "Time :: Time",  # 7th feature so 19 + 7 == 26, matching N_GENES comment above
]


def _make_synthetic_adata(n_obs: int = N_OBS, n_genes: int = N_GENES, seed: int = 0) -> ad.AnnData:
    """A stand-in for the LOOP h5ad, structurally identical to what the notebook reads.

    Reproduces the columns the downstream cells depend on (the 15 medium
    molecules, the 7 scatter features, and a `.X` whose width lands on the
    26-dimensional state the notebook prints). Values are log-normal draws on
    the scale each channel plausibly lives on, so `log1p` and standardization
    downstream are well-behaved. Expression is coupled to the medium with a
    small linear map, so the surrogate in the last cell has a signal to fit.
    """
    rng = np.random.default_rng(seed)
    molecule_medians = np.array(
        [
            1.0,  # 740-yp
            0.5,  # mtg
            10.0,  # rhflt3l
            5.0,  # gm-csf
            5.0,  # tpo
            50.0,  # sr1
            100.0,  # um171
            0.5,  # um729
            20.0,  # scf
            50.0,  # butyzamide
            1.0,  # retinoic_acid
            200.0,  # ldl
            5.0,  # il3
            20.0,  # o2
            12.0,  # days_of_culture
        ]
    )
    scatter_medians = np.array([1e5, 5e4, 1e2, 5e4, 2e4, 1e2, 1e1])

    obs = pd.DataFrame(index=pd.Index([f"cell_{i}" for i in range(n_obs)], name="cell_id"))
    obs["experiment_number"] = rng.integers(1, 5, size=n_obs)
    obs["replicate"] = rng.integers(1, 4, size=n_obs)
    # pandas dtypes go through pd.Categorical, not ndarray.astype("category"):
    # "category" is a pandas dtype and is not understood by NumPy arrays.
    obs["source_id"] = pd.Categorical(rng.choice(["src_a", "src_b", "src_c"], size=n_obs))
    obs["cell_type"] = pd.Categorical(rng.choice(["HSC", "MPP", "progenitor"], size=n_obs))

    for name, median in zip(MOLECULES, molecule_medians, strict=True):
        obs[name] = rng.lognormal(mean=np.log(median), sigma=0.4, size=n_obs)
    for name, median in zip(SCATTER_FEATURES, scatter_medians, strict=True):
        obs[name] = rng.lognormal(mean=np.log(median), sigma=0.3, size=n_obs)

    # Expression responds to the medium: each gene is a noisy linear readout
    # of the log-scale molecule vector. Gives the surrogate something to fit,
    # without pretending to be biology.
    W = rng.standard_normal((len(MOLECULES), n_genes)) * 0.3
    medium_log = np.log1p(obs[MOLECULES].to_numpy())
    signal = medium_log @ W
    X = np.exp(signal + rng.normal(0, 0.5, size=signal.shape)).astype(np.float32)

    var = pd.DataFrame(index=pd.Index([f"gene_{i}" for i in range(n_genes)], name="var_names"))
    return ad.AnnData(X=X, obs=obs, var=var)


if ADATA_PATH.exists():
    adata = ad.read_h5ad(ADATA_PATH)
    adata = adata[np.random.default_rng(0).choice(adata.n_obs, N_OBS, replace=False)].copy()
    print(f"loaded {N_OBS} cells from {ADATA_PATH}")
else:
    adata = _make_synthetic_adata()
    print(f"{ADATA_PATH} not found -- using a synthetic stand-in of the same shape")
adata

/Users/lorenzo.consoli/Work/data/collab-goettgens/loop0/data/adata_500k_fake.h5ad not found -- using a synthetic stand-in of the same shape


AnnData object with n_obs × n_vars = 500 × 19
    obs: 'experiment_number', 'replicate', 'source_id', 'cell_type', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'o2_[%]', 'days_of_culture', 'FSC-A :: FSC - Area', 'FSC-H :: FSC - Height', 'FSC-W :: FSC - Width', 'SSC-A :: SSC - Area', 'SSC-H :: SSC - Height', 'SSC-W :: SSC - Width', 'Time :: Time'
    layers: None (.X)

In [12]:
molecules = [
    "740-yp_[µm]",
    "mtg_[µm]",
    "rhflt3l_[ng_ml]",
    "gm-csf_[ng_ml]",
    "tpo_[ng_ml]",
    "sr1_[nm]",
    "um171_[nm]",
    "um729_[µm]",
    "scf_[ng_ml]",
    "butyzamide_[nm]",
    "retinoic_acid_[µm]",
    "ldl_[ng_ml]",
    "il3_[ng_ml]",
    "o2_[%]",
    "days_of_culture",
]
molecules_vals = np.log1p(adata.obs[molecules].astype(float).values)
condition_key = "medium_log1p"
adata.obsm[condition_key] = molecules_vals

In [13]:
scatter_features = [
    "FSC-A :: FSC - Area",
    "FSC-H :: FSC - Height",
    "FSC-W :: FSC - Width",
    "SSC-A :: SSC - Area",
    "SSC-H :: SSC - Height",
    "SSC-W :: SSC - Width",
]
X_scatter = adata.obs[scatter_features].values
X_joint = np.concatenate([adata.X, X_scatter], axis=1)

X_mean = X_joint.mean(0)
X_std = X_joint.std(0)
X = (X_joint - X_mean) / X_std

state_key = "X_std"
adata.obsm[state_key] = X

In [14]:
dm = DataManagerConfig(
    sample_rep=state_key,
    continuous_conditions=(condition_key,),
    # groups=["source_id"],
    # groups_encoding={"source_id": "one-hot"},
).build()
dmod = FlowDataModule.from_adata(
    adata,
    dm,
    n_train_steps=2_000,
    batch_size=128,
)
dmod.set_predict_data(
    adata,
    max_per_group=10_000,
    require_target_state=False,
)

In [15]:
module = MLPVelocity(
    dmod.data_dims.state_dim,
    time_features_id="torch-cfm",
    conditioning_id="resnet1d",
    vf_decoder_mlp_kwargs={"hidden_dims": [32, 32]},
    condition_encoder_input_layers={
        **{
            cond: {"input_dim": cond_dim, "output_dim": 2}
            for cond, cond_dim in dmod.data_dims.condition_continuous_dims.items()
        },
        **{
            cond: {"input_dim": cond_dim, "output_dim": 2} for cond, cond_dim in dmod.data_dims.groups_reps_dims.items()
        },
    },
)
inference = ODEInference(
    module,
    ODEInferenceConfig(generate_from_noise=True, n_samples=2),
    dims=dmod.data_dims,
)

adata_pred = predict_adata(dmod, inference, adata)
adata_pred

Predicting: 100%|██████████| 1/1 [00:00<00:00,  5.11it/s]


AnnData object with n_obs × n_vars = 500 × 26
    obs: '_sckitflow_all_cells'
    obsm: 'raw_samples', 'medium_log1p'
    layers: None (.X)

In [16]:
class MeanMarkerMSE(SurrogatePotential):
    """Class for mean marker regression."""

    def get_response(self, pred_data: PredictionData) -> torch.Tensor:
        """Computes the response by averaging over the generated samples."""
        return pred_data.raw_samples.mean(0)

    def compute_raw_potential(self, y: torch.Tensor, ystar: torch.Tensor):
        """Simple MSE loss against the target expression."""
        return torch.sum((y - ystar) ** 2, dim=-1)


ystar = torch.ones((X.shape[-1],))
potential = MeanMarkerMSE(inference, ystar, dmod)

N = 10
pot, grad_fn = jvp_out = torch.func.vjp(
    potential.forward,
    {condition_key: torch.zeros((N, len(molecules)))},
)
grad = grad_fn(torch.ones_like(pot))[0]["medium_log1p"]
print(pot.shape, grad.shape)

torch.Size([10]) torch.Size([10, 15])
